In [2]:
import torch
import time


# Helper to benchmark a function
def benchmark(func, warmup=5, reps=20):
    for _ in range(warmup):
        func()
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(reps):
        func()
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) / reps * 1000  # Return avg time in ms


if not torch.cuda.is_available():
    print("CUDA is not available. This benchmark requires a GPU.")
else:
    device = torch.device("cuda")
    print(f"Using device: {torch.cuda.get_device_name(0)}")

    # Common parameters
    batch_size = 4
    seq_len = 2048
    hidden_size = 2560  # From the paper's example

    # --- Case 1: Inefficient Shape (head_dim = 80) ---
    num_heads_inefficient = 32
    head_dim_inefficient = hidden_size // num_heads_inefficient  # 2560 / 32 = 80
    q_inefficient = torch.randn(
        batch_size * num_heads_inefficient,
        seq_len,
        head_dim_inefficient,
        device=device,
        dtype=torch.float16,
    )
    k_inefficient = torch.randn(
        batch_size * num_heads_inefficient,
        seq_len,
        head_dim_inefficient,
        device=device,
        dtype=torch.float16,
    )

    def inefficient_bmm():
        torch.bmm(q_inefficient, k_inefficient.transpose(1, 2))

    # --- Case 2: Efficient Shape (head_dim = 64) ---
    num_heads_efficient = 40
    head_dim_efficient = hidden_size // num_heads_efficient  # 2560 / 40 = 64
    q_efficient = torch.randn(
        batch_size * num_heads_efficient,
        seq_len,
        head_dim_efficient,
        device=device,
        dtype=torch.float16,
    )
    k_efficient = torch.randn(
        batch_size * num_heads_efficient,
        seq_len,
        head_dim_efficient,
        device=device,
        dtype=torch.float16,
    )

    def efficient_bmm():
        torch.bmm(q_efficient, k_efficient.transpose(1, 2))

    # Run benchmarks
    time_inefficient = benchmark(inefficient_bmm)
    time_efficient = benchmark(efficient_bmm)

    print("\n--- Testing Head Dimension (h/a) ---")
    print(f"Inefficient (h/a=80): {time_inefficient:.4f} ms")
    print(f"Efficient (h/a=64):   {time_efficient:.4f} ms")
    print(f"Speedup: {time_inefficient / time_efficient:.2f}x")

Using device: NVIDIA GeForce RTX 4090

--- Testing Head Dimension (h/a) ---
Inefficient (h/a=80): 1.3138 ms
Efficient (h/a=64):   1.6100 ms
Speedup: 0.82x


In [4]:
if torch.cuda.is_available():
    # Common parameters
    batch_size = 4
    seq_len = 2048
    ffn_dim_mult = 4

    # --- Case 1: Inefficient Hidden Size ---
    h_inefficient = 2560 + 1  # 2561, not divisible by 64
    input_tensor = torch.randn(
        batch_size * seq_len, h_inefficient, device=device, dtype=torch.float16
    )
    mlp_up = torch.randn(
        h_inefficient, ffn_dim_mult * h_inefficient, device=device, dtype=torch.float16
    )
    mlp_down = torch.randn(
        ffn_dim_mult * h_inefficient, h_inefficient, device=device, dtype=torch.float16
    )

    def inefficient_mlp():
        x = torch.matmul(input_tensor, mlp_up)
        x = torch.matmul(x, mlp_down)

    # --- Case 2: Efficient Hidden Size ---
    h_efficient = 2560  # Divisible by 64
    input_tensor_eff = torch.randn(
        batch_size * seq_len, h_efficient, device=device, dtype=torch.float16
    )
    mlp_up_eff = torch.randn(
        h_efficient, ffn_dim_mult * h_efficient, device=device, dtype=torch.float16
    )
    mlp_down_eff = torch.randn(
        ffn_dim_mult * h_efficient, h_efficient, device=device, dtype=torch.float16
    )

    def efficient_mlp():
        x = torch.matmul(input_tensor_eff, mlp_up_eff)
        x = torch.matmul(x, mlp_down_eff)

    # Run benchmarks
    time_inefficient = benchmark(inefficient_mlp)
    time_efficient = benchmark(efficient_mlp)

    print("\n--- Testing Hidden Dimension (h) ---")
    print(f"Inefficient (h=2561): {time_inefficient:.4f} ms")
    print(f"Efficient (h=2560):   {time_efficient:.4f} ms")
    print(f"Speedup: {time_inefficient / time_efficient:.2f}x")


--- Testing Hidden Dimension (h) ---
Inefficient (h=2561): 7.3698 ms
Efficient (h=2560):   5.3856 ms
Speedup: 1.37x


In [8]:
if torch.cuda.is_available():
    # Common parameters
    batch_size = 4
    seq_len = 2048
    hidden_size = 2560

    input_tensor = torch.randn(
        batch_size * seq_len, hidden_size, device=device, dtype=torch.float16
    )

    # --- Case 1: Inefficient Vocab Size ---
    v_inefficient = 50257  # A common vocab size, not divisible by 64
    lm_head_inefficient = torch.randn(
        hidden_size, v_inefficient, device=device, dtype=torch.float16
    )

    def inefficient_head():
        torch.matmul(input_tensor, lm_head_inefficient)

    # --- Case 2: Efficient Vocab Size ---
    v_efficient = 50304  # Padded to be a multiple of 64 (50257 -> 50304)
    lm_head_efficient = torch.randn(
        hidden_size, v_efficient, device=device, dtype=torch.float16
    )

    def efficient_head():
        torch.matmul(input_tensor, lm_head_efficient)

    # Run benchmarks
    time_inefficient = benchmark(inefficient_head)
    time_efficient = benchmark(efficient_head)

    print("\n--- Testing Vocabulary Size (v) ---")
    print(f"Inefficient (v=50257): {time_inefficient:.4f} ms")
    print(f"Efficient (v=50304):   {time_efficient:.4f} ms")
    print(f"Speedup: {time_inefficient / time_efficient:.2f}x")


--- Testing Vocabulary Size (v) ---
Inefficient (v=50257): 16.6932 ms
Efficient (v=50304):   13.2416 ms
Speedup: 1.26x


In [10]:
import torch
import time


# Helper to benchmark a function
def benchmark(func, warmup=5, reps=20):
    for _ in range(warmup):
        func()
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(reps):
        func()
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) / reps * 1000  # Return avg time in ms


if not torch.cuda.is_available():
    print("CUDA is not available. This benchmark requires a GPU.")
else:
    device = torch.device("cuda")
    print(f"Using device: {torch.cuda.get_device_name(0)}")

    # Common parameters (based on Llama-2 7B)
    batch_size = 2
    seq_len = 2048
    hidden_size = 4096  # h = 4096, a multiple of 64

    input_tensor = torch.randn(
        batch_size * seq_len, hidden_size, device=device, dtype=torch.float16
    )

    # --- Case 1: Inefficient Intermediate Size (Naive 8/3 ratio) ---
    inter_size_inefficient = int(hidden_size * 8 / 3)  # 10922, not divisible by 64
    w_gate_ineff = torch.randn(
        hidden_size, inter_size_inefficient, device=device, dtype=torch.float16
    )
    w_up_ineff = torch.randn(
        hidden_size, inter_size_inefficient, device=device, dtype=torch.float16
    )
    w_down_ineff = torch.randn(
        inter_size_inefficient, hidden_size, device=device, dtype=torch.float16
    )

    def inefficient_swiglu():
        gate = torch.matmul(input_tensor, w_gate_ineff)
        up = torch.matmul(input_tensor, w_up_ineff)
        fused = gate * torch.nn.functional.silu(up)  # SwiGLU logic
        output = torch.matmul(fused, w_down_ineff)

    # --- Case 2: Efficient Intermediate Size (LLaMA-2 7B's choice) ---
    inter_size_efficient = 11008  # Multiple of 128
    w_gate_eff = torch.randn(
        hidden_size, inter_size_efficient, device=device, dtype=torch.float16
    )
    w_up_eff = torch.randn(
        hidden_size, inter_size_efficient, device=device, dtype=torch.float16
    )
    w_down_eff = torch.randn(
        inter_size_efficient, hidden_size, device=device, dtype=torch.float16
    )

    def efficient_swiglu():
        gate = torch.matmul(input_tensor, w_gate_eff)
        up = torch.matmul(input_tensor, w_up_eff)
        fused = gate * torch.nn.functional.silu(up)  # SwiGLU logic
        output = torch.matmul(fused, w_down_eff)

    # Run benchmarks
    time_inefficient = benchmark(inefficient_swiglu)
    time_efficient = benchmark(efficient_swiglu)

    print("\n--- Testing SwiGLU Intermediate Dimension ---")
    print(f"Inefficient (dim={inter_size_inefficient}): {time_inefficient:.4f} ms")
    print(f"Efficient (dim={inter_size_efficient}):   {time_efficient:.4f} ms")
    print(f"Speedup: {time_inefficient / time_efficient:.2f}x")

Using device: NVIDIA GeForce RTX 4090

--- Testing SwiGLU Intermediate Dimension ---
Inefficient (dim=10922): 7.7746 ms
Efficient (dim=11008):   7.7533 ms
Speedup: 1.00x


In [12]:
import torch
import time

try:
    from flash_attn import flash_attn_func

    flash_attn_available = True
except ImportError:
    flash_attn_available = False
    print(
        "FlashAttention is not installed. Skipping Tip 5. To run, use: pip install flash-attn"
    )


# Helper to benchmark a function
def benchmark(func, warmup=5, reps=20):
    for _ in range(warmup):
        func()
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(reps):
        func()
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) / reps * 1000  # Return avg time in ms


if torch.cuda.is_available() and flash_attn_available:
    device = torch.device("cuda")
    print(f"Using device: {torch.cuda.get_device_name(0)}")

    # Inefficient shape from the paper's main example
    batch_size = 4
    seq_len = 2048
    hidden_size = 2560  # h
    num_heads = 32  # a
    head_dim = hidden_size // num_heads  # h/a = 80 (inefficient)

    q = torch.randn(
        batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16
    )
    k = torch.randn(
        batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16
    )
    v = torch.randn(
        batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16
    )

    # --- Case 1: Standard PyTorch Attention (with inefficient head_dim) ---
    def standard_attention():
        q_std = (
            q.permute(0, 2, 1, 3)
            .contiguous()
            .view(batch_size * num_heads, seq_len, head_dim)
        )
        k_std = (
            k.permute(0, 2, 1, 3)
            .contiguous()
            .view(batch_size * num_heads, seq_len, head_dim)
        )
        v_std = (
            v.permute(0, 2, 1, 3)
            .contiguous()
            .view(batch_size * num_heads, seq_len, head_dim)
        )

        attn_scores = torch.bmm(q_std, k_std.transpose(1, 2)) / (head_dim**0.5)
        attn_probs = torch.nn.functional.softmax(attn_scores, dim=-1)
        output = torch.bmm(attn_probs, v_std)

    # --- Case 2: FlashAttention (with the same inefficient head_dim) ---
    q_flash = q.view(batch_size, seq_len, -1)
    k_flash = k.view(batch_size, seq_len, -1)
    v_flash = v.view(batch_size, seq_len, -1)

    def flash_attention_impl():
        # flash_attn_func expects (batch_size, seq_len, num_heads, head_dim)
        output = flash_attn_func(q, k, v, causal=True)

    # Run benchmarks
    time_standard = benchmark(standard_attention)
    time_flash = benchmark(flash_attention_impl)

    print("\n--- Testing Standard vs. FlashAttention with Inefficient head_dim=80 ---")
    print(f"Standard Attention: {time_standard:.4f} ms")
    print(f"FlashAttention:     {time_flash:.4f} ms")
    print(f"Speedup from FlashAttention: {time_standard / time_flash:.2f}x")

Using device: NVIDIA GeForce RTX 4090

--- Testing Standard vs. FlashAttention with Inefficient head_dim=80 ---
Standard Attention: 7.5777 ms
FlashAttention:     0.7956 ms
Speedup from FlashAttention: 9.52x
